# Stage 2 — Finance Instruction Fine-Tuning (SFT)
This standalone notebook fine-tunes a Finance FAQ Assistant on verified instruction–response examples. It includes base-model evaluation, Alpaca-style prompt formatting, QLoRA SFT, before/after evaluation, adapter saving, and a Finance-only ipywidgets demo.

In [1]:
# Install pinned, Colab-compatible packages
!pip install -q "unsloth[colab-new]" "trl>=0.18,<0.26" "transformers>=4.51,<5" datasets accelerate bitsandbytes ipywidgets pandas matplotlib


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 34.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 126.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 45.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 118.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.3/322.3 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
from pathlib import Path
import json, warnings, torch
from datasets import Dataset
warnings.filterwarnings("ignore")

DOMAIN = "Finance FAQ Assistant"
DOMAIN_OPTIONS = [DOMAIN]
BASE_MODEL = "unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 1024
DTYPE = None
LOAD_IN_4BIT = True
SEED = 42

print("Domain:", DOMAIN)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU detected")


Domain: Finance FAQ Assistant
GPU: Tesla T4


In [3]:
# Locate extracted data directory (no ZIP upload required).
candidates = [
    Path("../data"),                # local run from notebooks/
    Path("./data"),                 # local run from project root
    Path("/content/data"),          # Colab with uploaded/extracted data folder
    Path("/data"),                  # absolute data mount
    Path("/content/finance_stage1_data"),  # backward compatibility
    Path("/content/finance_stage2_data"),
    Path("/content/finance_stage3_data"),
]

DATA_DIR = next((p for p in candidates if p.exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError(
        "Could not find extracted data folder. Expected one of: " + ", ".join(str(p) for p in candidates)
    )

DATA_DIR = DATA_DIR.resolve()
print("Using DATA_DIR:", DATA_DIR)
print("Available files:", sorted(p.name for p in DATA_DIR.glob("*") if p.is_file()))


Using DATA_DIR: /content/data
Available files: ['instruction_dataset.jsonl']


In [4]:
rows=[json.loads(x) for x in (DATA_DIR/"instruction_dataset.jsonl").read_text(encoding="utf-8").splitlines() if x.strip()]
print("Instruction examples:",len(rows))
def format_row(r):
    return {"text":f"### Instruction:\nYou are a professional Finance FAQ Assistant. Provide accurate, safe, clear educational information.\n\n### User:\n{r['instruction'].strip()}\n\n### Assistant:\n{r['response'].strip()}{tokenizer.eos_token if 'tokenizer' in globals() else ''}"}


Instruction examples: 105


In [5]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)
print("Model and QLoRA adapter initialized.")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.7.2: Fast Qwen2 patching. Transformers: 4.57.6.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth 2026.7.2 patched 24 layers with 24 QKV layers, 24 O layers and 24 MLP layers.


Model and QLoRA adapter initialized.


In [6]:
def format_row(r):
    return {"text":f"### Instruction:\nYou are a professional Finance FAQ Assistant. Provide accurate, safe, clear educational information.\n\n### User:\n{r['instruction'].strip()}\n\n### Assistant:\n{r['response'].strip()}{tokenizer.eos_token}"}
train_dataset=Dataset.from_list([format_row(r) for r in rows])
print(train_dataset[0]["text"][:800])

### Instruction:
You are a professional Finance FAQ Assistant. Provide accurate, safe, clear educational information.

### User:
What is a savings account?

### Assistant:
A savings account is a basic banking product that allows you to deposit money safely while earning interest. Interest rates typically range from 3% to 7% per annum. It is designed for individuals who want to keep their money secure and earn a modest return over time.<|im_end|>


## Step 7 - Base vs SFT comparison (same 10 questions)

In [7]:
# Capture base answers for Step 7 using the same 10 questions used in Step 5.
EVAL_QUESTIONS = [
    "What is a savings account?",
    "What is the difference between savings and current account?",
    "What documents are required for KYC?",
    "How do I transfer money using NEFT?",
    "What is RTGS and when should I use it?",
    "What is a credit score and why does it matter?",
    "How can I apply for a personal loan?",
    "What should I do if I receive a suspicious OTP request?",
    "My EMI payment failed. What should I do?",
    "How can I improve my credit score?",
]

def ask_model(question, max_new_tokens=240, temperature=0.2):
    prompt = f"### Instruction:\nYou are a professional Finance FAQ Assistant. Provide accurate, safe, clear educational information.\n\n### User:\n{question}\n\n### Assistant:\n"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH).to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=temperature > 0,
        temperature=max(temperature, 0.01),
        top_p=0.9,
        repetition_penalty=1.08,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

FastLanguageModel.for_inference(model)
base_answers = {q: ask_model(q) for q in EVAL_QUESTIONS}
for q, a in base_answers.items():
    print("\nQ:", q, "\nBase:", a)


Q: What is a savings account? 
Base: A savings account is a type of bank account that offers you the opportunity to save money for future use, such as buying a car or paying off debt. Savings accounts typically offer higher interest rates than checking accounts and can be used to invest in stocks, bonds, and other financial instruments. They are also often offered with lower fees compared to checking accounts. It's important to note that it's always best to consult with a financial advisor before opening any new accounts. 

Savings accounts are available at banks, credit unions, and online banking platforms. You can open an account by visiting your local branch, calling 211, or using the online banking platform. The account will have a unique account number, which you can share with friends and family members who want to open an account on your behalf. 

Remember, it's important to understand the risks associated with saving accounts, including the possibility of losing money if you w

## Train Stage 2 SFT

In [8]:
from trl import SFTTrainer, SFTConfig
OUTPUT_DIR=Path("/content/finance_adapters/stage2_sft")
args=SFTConfig(output_dir=str(OUTPUT_DIR),dataset_text_field="text",max_length=MAX_SEQ_LENGTH,per_device_train_batch_size=1,gradient_accumulation_steps=4,num_train_epochs=2,learning_rate=2e-4,warmup_ratio=0.05,logging_steps=5,save_strategy="no",report_to="none",optim="adamw_8bit",seed=SEED)
trainer=SFTTrainer(model=model,tokenizer=tokenizer,train_dataset=train_dataset,args=args)
trainer.train(); model.save_pretrained(str(OUTPUT_DIR)); tokenizer.save_pretrained(str(OUTPUT_DIR))
print("Saved:",OUTPUT_DIR)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/105 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 105 | Num Epochs = 2 | Total steps = 54
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 8,798,208 of 502,830,976 (1.75% trained)


Step,Training Loss
5,2.890600
10,1.934200
15,1.606800
20,1.590600
25,1.372100
30,1.384100
35,1.223600
40,1.105000
45,1.203500
50,1.249200


Saved: /content/finance_adapters/stage2_sft


## Step 7 - Comparison table and report export

In [9]:
import re
from pathlib import Path
import pandas as pd
from IPython.display import display

def resolve_reports_dir():
    candidates = []
    if "DATA_DIR" in globals():
        candidates.append(Path(DATA_DIR).resolve().parent / "reports")
    candidates.extend([Path.cwd() / "reports", Path("/content/reports"), Path("./reports"), Path("../reports")])

    for p in candidates:
        try:
            p.mkdir(parents=True, exist_ok=True)
            probe = p / ".write_probe"
            probe.write_text("ok", encoding="utf-8")
            probe.unlink()
            return p.resolve()
        except Exception:
            continue
    raise RuntimeError("Could not create a writable reports directory.")

def score_answer(question, answer):
    q = question.lower()
    a = answer.lower().strip()
    words = re.findall(r"[a-zA-Z]+", a)
    score = 0
    notes = []

    # Length/detail
    if len(words) >= 30:
        score += 2
    elif len(words) >= 18:
        score += 1
    else:
        notes.append("too brief")

    # Domain specificity
    domain_terms = [
        "kyc", "neft", "rtgs", "imps", "upi", "emi", "cibil",
        "interest", "loan", "deposit", "bank", "account", "ifsc", "otp"
    ]
    domain_hits = sum(1 for t in domain_terms if t in a)
    if domain_hits >= 2:
        score += 2
    elif domain_hits == 1:
        score += 1
    else:
        notes.append("low domain specificity")

    # Safety behavior for fraud/OTP questions
    if "otp" in q or "unauthorized" in q or "suspicious" in q:
        if any(k in a for k in ["never share", "report", "fraud", "block", "cybercrime", "helpline"]):
            score += 2
        else:
            notes.append("missing safety guidance")

    # Procedural clarity for how-to questions
    if any(k in q for k in ["how do i", "how can i", "what should i do"]):
        if any(k in a for k in ["log in", "visit", "submit", "documents", "steps", "contact", "immediately"]):
            score += 1
        else:
            notes.append("weak actionable steps")

    # Penalize vague phrasing
    if any(k in a for k in ["it depends", "usually", "generally", "contact your bank"]):
        score -= 1
        notes.append("generic phrasing")

    return score, notes

def compare_answers(question, base_ans, sft_ans):
    base_score, base_notes = score_answer(question, base_ans)
    sft_score, sft_notes = score_answer(question, sft_ans)

    if sft_score > base_score:
        better = "Fine-Tuned Model"
        reason = f"More domain-specific and actionable (score {sft_score} vs {base_score})."
    elif base_score > sft_score:
        better = "Base Model"
        reason = f"More complete/safe for this question (score {base_score} vs {sft_score})."
    else:
        better = "Tie"
        reason = f"Comparable quality (score {base_score} each)."

    if base_notes or sft_notes:
        base_hint = ", ".join(base_notes[:2]) if base_notes else "none"
        sft_hint = ", ".join(sft_notes[:2]) if sft_notes else "none"
        reason += f" Notes -> Base: {base_hint}; SFT: {sft_hint}."

    return better, reason, base_score, sft_score

FastLanguageModel.for_inference(model)
sft_answers = {q: ask_model(q) for q in EVAL_QUESTIONS}

rows = []
wins = {"Fine-Tuned Model": 0, "Base Model": 0, "Tie": 0}
score_gains = []

for q in EVAL_QUESTIONS:
    better, reason, base_score, sft_score = compare_answers(q, base_answers[q], sft_answers[q])
    wins[better] += 1
    score_gains.append(sft_score - base_score)
    rows.append({
        "Question": q,
        "Base Model Answer": base_answers[q],
        "Fine-Tuned Model Answer": sft_answers[q],
        "Which Is Better?": better,
        "Reason": reason,
    })

df_sft = pd.DataFrame(rows)
display(df_sft)

# Export Step 7 table to reports/sft_model_comparison.md
reports_dir = resolve_reports_dir()
out_path = reports_dir / "sft_model_comparison.md"

lines = [
    "# Base vs SFT Comparison (Finance FAQ Assistant)",
    "",
    "This report compares responses from the base model and instruction fine-tuned (SFT) model on the same 10 domain questions.",
    "",
    "## Evaluation Criteria",
    "",
    "- Correctness",
    "- Domain accuracy",
    "- Clarity",
    "- Safety",
    "- Helpfulness",
    "- Less generic response",
    "- Better domain-specific behavior",
    "",
    "## Comparison Table",
    "",
    "| Question | Base Model Answer | Fine-Tuned Model Answer | Which Is Better? | Reason |",
    "| --- | --- | --- | --- | --- |",
]
for row in rows:
    q = " ".join(row["Question"].split()).replace("|", "\\|")
    base = " ".join(row["Base Model Answer"].split()).replace("|", "\\|")
    sft = " ".join(row["Fine-Tuned Model Answer"].split()).replace("|", "\\|")
    better = row["Which Is Better?"].replace("|", "\\|")
    reason = " ".join(row["Reason"].split()).replace("|", "\\|")
    lines.append(f"| {q} | {base} | {sft} | {better} | {reason} |")

avg_gain = sum(score_gains) / len(score_gains) if score_gains else 0.0
lines += [
    "",
    "## SFT Takeaways",
    "",
    f"- Win count -> Fine-Tuned: {wins['Fine-Tuned Model']}, Base: {wins['Base Model']}, Tie: {wins['Tie']}",
    f"- Average score gain (SFT - Base): {avg_gain:.2f}",
    "- Typical SFT improvements: better domain specificity, clearer steps, and improved finance safety tone.",
]

out_path.write_text("\n".join(lines), encoding="utf-8")
print("Step 7 report written:", out_path)

,Question,Base Model Answer,Fine-Tuned Model Answer,Which Is Better?,Reason
0,What is a savings account?,A savings account is a type of bank account th...,A savings account is a deposit account that ea...,Tie,Comparable quality (score 4 each).
1,What is the difference between savings and cur...,Savings refers to an investment of money that ...,Savings accounts offer higher interest rates o...,Fine-Tuned Model,More domain-specific and actionable (score 4 v...
2,What documents are required for KYC?,"To initiate the KYC process, you will need to ...",KYC is mandatory for all financial transaction...,Tie,Comparable quality (score 4 each).
3,How do I transfer money using NEFT?,NEFT (National Electronic Funds Transfer) is a...,NEFT transfers use the NEFT network and are av...,Base Model,More complete/safe for this question (score 5 ...
4,What is RTGS and when should I use it?,RTGS stands for Real-Time Gross Settlement. It...,RTGS (Real Time Gross Settlement) is a real-ti...,Tie,Comparable quality (score 4 each).
5,What is a credit score and why does it matter?,A credit score is a numerical value assigned t...,"A credit score ranges from 300 to 900, with 85...",Base Model,More complete/safe for this question (score 4 ...
6,How can I apply for a personal loan?,Certainly! Here's how you can apply for a pers...,Apply online through the bank's website or mob...,Tie,Comparable quality (score 5 each).
7,What should I do if I receive a suspicious OTP...,If you receive an OTP request from a financial...,If you receive an OTP request from a financial...,Tie,Comparable quality (score 7 each).
8,My EMI payment failed. What should I do?,Certainly! Here's what you need to do:\n\n1. *...,"If your EMI payment fails, contact the bank im...",Tie,Comparable quality (score 4 each). Notes -> Ba...
9,How can I improve my credit score?,Improving your credit score is an important st...,"To improve your credit score, pay all bills on...",Base Model,More complete/safe for this question (score 5 ...


Step 7 report written: /content/reports/sft_model_comparison.md


## Interactive demo

In [10]:
# Finance-only ipywidgets demo
import html, ipywidgets as widgets
from IPython.display import display, HTML

FastLanguageModel.for_inference(model)

domain_dd = widgets.Dropdown(options=DOMAIN_OPTIONS, value=DOMAIN, description="Domain", style={"description_width":"100px"}, layout=widgets.Layout(width="580px"))
question_box = widgets.Textarea(value="What is the difference between NEFT and RTGS?", description="Question", style={"description_width":"100px"}, layout=widgets.Layout(width="900px", height="110px"))
max_tokens = widgets.IntSlider(value=220, min=80, max=400, step=20, description="Max tokens", style={"description_width":"100px"}, layout=widgets.Layout(width="580px"))
temperature = widgets.FloatSlider(value=0.2, min=0.1, max=0.8, step=0.05, description="Temperature", style={"description_width":"100px"}, layout=widgets.Layout(width="580px"))
button = widgets.Button(description="Generate Finance Answer", button_style="primary", layout=widgets.Layout(width="260px", height="44px"))
status = widgets.HTML()
answer = widgets.HTML(value="<div style='padding:18px;border:1px solid #cbd5e1;border-radius:14px;background:white'>Answer will appear here.</div>")

def render(text):
    return f"<div style='padding:22px;border:1px solid #bfdbfe;border-radius:16px;background:linear-gradient(180deg,#fff,#eff6ff);line-height:1.7'><b style='color:#1e3a8a'>Finance Assistant Answer</b><hr>{html.escape(text).replace(chr(10),'<br>')}</div>"

def on_click(_):
    q=question_box.value.strip()
    if not q:
        answer.value=render("Please enter a finance-related question.")
        return
    button.disabled=True; status.value="<b style='color:#2563eb'>Generating...</b>"
    try:
        torch.cuda.empty_cache()
        answer.value=render(ask_model(q, max_new_tokens=max_tokens.value, temperature=temperature.value))
        status.value="<b style='color:#15803d'>Completed</b>"
    except Exception as exc:
        answer.value=render(f"{type(exc).__name__}: {exc}")
        status.value="<b style='color:#b91c1c'>Failed</b>"
    finally:
        button.disabled=False
button.on_click(on_click)

display(HTML("""<div style='background:linear-gradient(135deg,#0f172a,#1d4ed8);color:white;padding:26px;border-radius:18px;text-align:center;margin-bottom:18px'><h1>Enterprise Finance FAQ Assistant</h1><p>Standalone fine-tuning stage demonstration</p></div>"""))
display(domain_dd, max_tokens, temperature, question_box, button, status, answer)


Dropdown(description='Domain', layout=Layout(width='580px'), options=('Finance FAQ Assistant',), style=Descrip…

IntSlider(value=220, description='Max tokens', layout=Layout(width='580px'), max=400, min=80, step=20, style=S…

FloatSlider(value=0.2, description='Temperature', layout=Layout(width='580px'), max=0.8, min=0.1, step=0.05, s…

Textarea(value='What is the difference between NEFT and RTGS?', description='Question', layout=Layout(height='…

Button(button_style='primary', description='Generate Finance Answer', layout=Layout(height='44px', width='260p…

HTML(value='')

HTML(value="<div style='padding:18px;border:1px solid #cbd5e1;border-radius:14px;background:white'>Answer will…